# Phase 3: File-Level Feature Extraction & Logic-Based Detection
## Test Notebook - 12-Kimsuky Dataset

### Objective
Transform event-level data from Phase 2 into file-level features with explicit detection flags based on Oh et al. algorithms.

### Detection Logic Scope (STRICT)
**Core Logic:**
- LogFile-1A (Algorithms 1-4): Timestamp change extraction and validation
- UsnJrnl-1A (Algorithms 5-7): BASIC_INFO_CHANGE pattern detection

**Additional Validation:**
- LogFile-4 (Algorithm 10): $FN timestamp manipulation via file move
- UsnJrnl-3 (Algorithm 11): $FN manipulation via USN patterns

### Input
- grouped_events_12-Kimsuky.csv (from Phase 2)

### Outputs
- phase3_file_features_12_Kimsuky.csv (one row per FileFRN)
- phase3_detection_flags_12_Kimsuky.csv (one row per FileFRN with boolean flags)

### Key Constraint
This phase is LOGIC-FIRST, not ML-first. Each feature and flag must be forensically justified and mapped to Oh et al. algorithms.


In [21]:
# [Cell 2] Imports and Configuration

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuration
DATASET_NAME = "03-PE"
DATA_ID = "03-PE"

# Input path (Phase 2 output)
INPUT_PATH = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 2: Data Preprocessing & Event Grouping/Test Notebook Outputs/grouped_events_03-PE.csv")

# Output paths
OUTPUT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 3: Feature Engineering & Labeling/Test Notebook Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_OUTPUT = OUTPUT_DIR / "phase3_file_features_03-PE.csv"
FLAGS_OUTPUT = OUTPUT_DIR / "phase3_detection_flags_03-PE.csv"

print(f"Dataset: {DATASET_NAME}")
print(f"Input: {INPUT_PATH}")
print(f"Output directory: {OUTPUT_DIR}")


Dataset: 03-PE
Input: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 2: Data Preprocessing & Event Grouping/Test Notebook Outputs/grouped_events_03-PE.csv
Output directory: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 3: Feature Engineering & Labeling/Test Notebook Outputs


In [22]:
# [Cell 3] Load Phase 2 Grouped Events

print("Loading Phase 2 grouped events...")
df = pd.read_csv(INPUT_PATH, low_memory=False)

print(f"Total events: {len(df):,}")
print(f"Unique files (FileFRN): {df['FileFRN'].nunique():,}")
print(f"Event sources: {df['EventSource'].value_counts().to_dict()}")

# Parse timestamp columns
timestamp_cols = [
    'EventTimestamp',
    'Undo_$SI-C', 'Undo_$SI-M', 'Undo_$SI-E', 'Undo_$SI-A',
    'Redo_$SI-C', 'Redo_$SI-M', 'Redo_$SI-E', 'Redo_$SI-A'
]

for col in timestamp_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print("\nColumn types after parsing:")
print(df.dtypes)


Loading Phase 2 grouped events...
Total events: 375,137
Unique files (FileFRN): 109,985
Event sources: {'UsnJrnl': 245425, 'LogFile': 129712}

Column types after parsing:
dataID                        object
FileFRN                      float64
FileName                      object
FilePath                      object
EventSource                   object
EventTimestamp        datetime64[ns]
LSN                          float64
USN                          float64
RedoOP                       float64
UndoOP                       float64
RedoOPName                    object
UndoOPName                    object
RecordOffset                 float64
AttributeOffset              float64
TargetVCN                    float64
IsTimestampChange               bool
Undo_$SI-C            datetime64[ns]
Undo_$SI-M            datetime64[ns]
Undo_$SI-E            datetime64[ns]
Undo_$SI-A            datetime64[ns]
Redo_$SI-C            datetime64[ns]
Redo_$SI-M            datetime64[ns]
Redo_$SI-E     

## Feature Engineering Categories

### A. Timestamp Change Features (LogFile-1A, Algorithm 3)
Derived from comparing Redo vs Undo timestamps in $LogFile records.

- `num_timestamp_changes`: Count of IsTimestampChange=True events
- `num_backward_jumps`: Count where Redo_$SI-* < Undo_$SI-* (timestamp moved to past)
- `num_forward_jumps`: Count where Redo_$SI-* > Undo_$SI-* (timestamp moved to future)
- `max_backward_jump_seconds`: Largest backward time delta
- `mean_jump_seconds`: Average absolute time delta
- `timestamp_change_density`: Timestamp changes per event for this file

### B. Structural NTFS Patterns (Algorithm 1-2)
Patterns indicating suspicious $SI manipulation.

- `only_SI_modified`: No $FN changes detected (inferred from attribute offsets)
- `repeated_update_resident_value`: Multiple UpdateResidentValue operations
- `consecutive_timestamp_changes`: Sequential timestamp change events without gaps

### C. Cross-Artifact Consistency (Algorithms 5-7)
Correlation between LogFile and UsnJrnl evidence.

- `has_logfile_ts_change`: Any LogFile timestamp change for this file
- `has_usn_basic_info`: Any BASIC_INFO_CHANGE in UsnJrnl for this file
- `has_usn_close`: Any CLOSE event in UsnJrnl following BASIC_INFO_CHANGE
- `logfile_usn_mismatch`: Timestamp change without corresponding USN evidence

### D. Temporal Behavior Features
Time-based patterns that may indicate automation or tools.

- `min_inter_event_delta`: Shortest time between consecutive events
- `max_inter_event_delta`: Longest time between consecutive events
- `burstiness_score`: Concentration of timestamp changes in short windows


In [23]:
# [Cell 5] Helper Functions for Timestamp Analysis
# Reference: Oh et al. Algorithm 3 - Checking Timestamp Changes

def compute_timestamp_jump(undo_ts, redo_ts):
    """
    Compute the time delta between Undo (before) and Redo (after) timestamps.
    
    Algorithm 3 Logic:
    - If before_$SI-C != after_$SI-C: potential manipulation
    - If before_$SI-M > after_$SI-M: backward jump (suspicious)
    
    Returns: delta in seconds (negative = backward jump, positive = forward jump)
    """
    if pd.isna(undo_ts) or pd.isna(redo_ts):
        return np.nan
    delta = (redo_ts - undo_ts).total_seconds()
    return delta

def has_zero_nanoseconds(timestamp):
    """
    Check if timestamp has zero nanoseconds component.
    
    Algorithm 3 Logic:
    - "if event.after_$SI-C.nanosecond == 0" indicates additional detection factor
    - Many timestomping tools (SetFileTime, PowerShell) produce zero nanoseconds
    
    Returns: True if nanoseconds are zero or timestamp is at exact second boundary
    """
    if pd.isna(timestamp):
        return False
    # Check if microseconds are zero (pandas resolution)
    return timestamp.microsecond == 0

def is_backward_jump(undo_ts, redo_ts):
    """
    Detect backward timestamp jump (timestamp moved to the past).
    
    Algorithm 3 Logic:
    - "event.before_$SI-M > event.after_$SI-M" indicates backward manipulation
    
    Returns: True if Redo timestamp is earlier than Undo timestamp
    """
    if pd.isna(undo_ts) or pd.isna(redo_ts):
        return False
    return redo_ts < undo_ts

def is_forward_jump(undo_ts, redo_ts):
    """
    Detect forward timestamp jump (timestamp moved to the future).
    Normal file operations typically produce small forward jumps.
    Large forward jumps may indicate manipulation.
    
    Returns: True if Redo timestamp is later than Undo timestamp
    """
    if pd.isna(undo_ts) or pd.isna(redo_ts):
        return False
    return redo_ts > undo_ts

print("Timestamp analysis helper functions defined.")
print("- compute_timestamp_jump(): Calculate delta in seconds")
print("- has_zero_nanoseconds(): Detect zero-nanosecond patterns")
print("- is_backward_jump(): Detect backward timestamp manipulation")
print("- is_forward_jump(): Detect forward timestamp changes")


Timestamp analysis helper functions defined.
- compute_timestamp_jump(): Calculate delta in seconds
- has_zero_nanoseconds(): Detect zero-nanosecond patterns
- is_backward_jump(): Detect backward timestamp manipulation
- is_forward_jump(): Detect forward timestamp changes


In [24]:
# [Cell 6] Compute Per-Event Indicators
# These will be aggregated to file-level features

# Identify LogFile timestamp change events
df['is_logfile_event'] = df['EventSource'] == 'LogFile'
df['is_usnjrnl_event'] = df['EventSource'] == 'UsnJrnl'

# For timestamp change events, compute jumps for each timestamp type
# Using $SI-M (Modified) as primary indicator per Algorithm 3
df['backward_jump_M'] = df.apply(
    lambda row: is_backward_jump(row['Undo_$SI-M'], row['Redo_$SI-M']) 
    if row['IsTimestampChange'] else False, axis=1
)

df['forward_jump_M'] = df.apply(
    lambda row: is_forward_jump(row['Undo_$SI-M'], row['Redo_$SI-M']) 
    if row['IsTimestampChange'] else False, axis=1
)

# Compute jump magnitude in seconds
df['jump_seconds_M'] = df.apply(
    lambda row: compute_timestamp_jump(row['Undo_$SI-M'], row['Redo_$SI-M'])
    if row['IsTimestampChange'] else np.nan, axis=1
)

# Also check Creation time changes (Algorithm 3: before_$SI-C != after_$SI-C)
df['creation_changed'] = df.apply(
    lambda row: (not pd.isna(row['Undo_$SI-C']) and 
                 not pd.isna(row['Redo_$SI-C']) and 
                 row['Undo_$SI-C'] != row['Redo_$SI-C'])
    if row['IsTimestampChange'] else False, axis=1
)

# Check for zero nanoseconds in Redo timestamps (Algorithm 3 additional factor)
df['redo_zero_ns_C'] = df['Redo_$SI-C'].apply(has_zero_nanoseconds)
df['redo_zero_ns_M'] = df['Redo_$SI-M'].apply(has_zero_nanoseconds)
df['redo_zero_ns_E'] = df['Redo_$SI-E'].apply(has_zero_nanoseconds)
df['redo_zero_ns_A'] = df['Redo_$SI-A'].apply(has_zero_nanoseconds)

# Any zero nanoseconds in Redo timestamps
df['has_redo_zero_ns'] = (df['redo_zero_ns_C'] | df['redo_zero_ns_M'] | 
                          df['redo_zero_ns_E'] | df['redo_zero_ns_A'])

print("Per-event indicators computed:")
print(f"  Backward jumps (M): {df['backward_jump_M'].sum():,}")
print(f"  Forward jumps (M): {df['forward_jump_M'].sum():,}")
print(f"  Creation time changes: {df['creation_changed'].sum():,}")
print(f"  Events with zero nanoseconds: {df['has_redo_zero_ns'].sum():,}")


Per-event indicators computed:
  Backward jumps (M): 89
  Forward jumps (M): 7,629
  Creation time changes: 63
  Events with zero nanoseconds: 31


## File-Level Feature Aggregation

Now we aggregate per-event indicators to file-level features.
Each FileFRN becomes one row in the output.

### Aggregation Strategy
- **Counts**: Sum of boolean indicators
- **Statistics**: Min/max/mean of numeric values
- **Existence flags**: Any True value → True for file


In [25]:
# [Cell 8] Aggregate Timestamp Change Features (Category A)
# Reference: Oh et al. Algorithm 3 - Checking Timestamp Changes

def aggregate_timestamp_features(group):
    """
    Aggregate timestamp change features for a single file (FileFRN group).
    
    Algorithm 3 references:
    - num_timestamp_changes: Count of detected changes
    - backward/forward jumps: Directional analysis
    - zero nanoseconds: Additional detection factor
    """
    ts_events = group[group['IsTimestampChange'] == True]
    total_events = len(group)
    
    features = {
        # A.1 Count of timestamp changes (Algorithm 1: extraction)
        'num_timestamp_changes': len(ts_events),
        
        # A.2 Backward jumps (Algorithm 3: before_$SI-M > after_$SI-M)
        'num_backward_jumps': group['backward_jump_M'].sum(),
        
        # A.3 Forward jumps
        'num_forward_jumps': group['forward_jump_M'].sum(),
        
        # A.4 Creation time changes (Algorithm 3: before_$SI-C != after_$SI-C)
        'num_creation_changes': group['creation_changed'].sum(),
        
        # A.5 Maximum backward jump magnitude
        'max_backward_jump_seconds': abs(group.loc[group['backward_jump_M'], 'jump_seconds_M'].min()) 
            if group['backward_jump_M'].any() else 0,
        
        # A.6 Mean jump magnitude
        'mean_jump_seconds': group['jump_seconds_M'].abs().mean() 
            if group['jump_seconds_M'].notna().any() else 0,
        
        # A.7 Timestamp change density
        'timestamp_change_density': len(ts_events) / total_events if total_events > 0 else 0,
        
        # A.8 Zero nanoseconds count (Algorithm 3: additional_detection_factor)
        'num_zero_nanosecond_events': group['has_redo_zero_ns'].sum(),
    }
    
    return pd.Series(features)

# Group by FileFRN and aggregate
print("Aggregating timestamp change features per file...")
ts_features = df.groupby('FileFRN').apply(aggregate_timestamp_features).reset_index()

print(f"Files with features: {len(ts_features):,}")
print(f"\nFeature columns: {list(ts_features.columns)}")

# Preview files with backward jumps
backward_files = ts_features[ts_features['num_backward_jumps'] > 0]
print(f"\nFiles with backward jumps: {len(backward_files):,}")


Aggregating timestamp change features per file...
Files with features: 109,985

Feature columns: ['FileFRN', 'num_timestamp_changes', 'num_backward_jumps', 'num_forward_jumps', 'num_creation_changes', 'max_backward_jump_seconds', 'mean_jump_seconds', 'timestamp_change_density', 'num_zero_nanosecond_events']

Files with backward jumps: 58


In [26]:
# [Cell 9] Aggregate Structural NTFS Features (Category B)
# Reference: Oh et al. Algorithm 1-2

def aggregate_structural_features(group):
    """
    Aggregate structural NTFS pattern features.
    
    Algorithm 1 references:
    - Record offset 0x38 = $STANDARD_INFORMATION attribute
    - Attribute offset 0x18-0x30 = timestamp fields within $SI
    
    Algorithm 2 references:
    - Target file identification via LSN or VCN+cluster index
    """
    logfile_events = group[group['is_logfile_event']]
    ts_events = group[group['IsTimestampChange'] == True]
    
    features = {
        # B.1 Only $SI modified (no $FN changes)
        # Inferred from attribute offset - offset 0x18-0x30 are $SI timestamps
        # $FN would have different offsets (typically in $FILE_NAME attribute)
        'only_SI_modified': (
            logfile_events['AttributeOffset'].isin([24, 32, 40, 48]).all() 
            if len(logfile_events) > 0 else False
        ),
        
        # B.2 Repeated UpdateResidentValue operations (Algorithm 1: redo.op == 0x7)
        'num_update_resident_value': (
            (logfile_events['RedoOPName'] == 'UpdateResidentValue').sum()
        ),
        'repeated_update_resident_value': (
            (logfile_events['RedoOPName'] == 'UpdateResidentValue').sum() > 1
        ),
        
        # B.3 Consecutive timestamp changes
        # Check if timestamp change events are adjacent in LSN order
        'consecutive_timestamp_changes': False,  # Will compute below
        
        # B.4 Total LogFile events for this file
        'num_logfile_events': len(logfile_events),
    }
    
    # Compute consecutive timestamp changes
    if len(ts_events) >= 2:
        ts_lsns = ts_events['LSN'].dropna().sort_values()
        if len(ts_lsns) >= 2:
            # Check if any LSNs are within 1000 of each other (close sequence)
            lsn_diffs = ts_lsns.diff().dropna()
            features['consecutive_timestamp_changes'] = (lsn_diffs < 1000).any()
    
    return pd.Series(features)

print("Aggregating structural NTFS features per file...")
struct_features = df.groupby('FileFRN').apply(aggregate_structural_features).reset_index()

print(f"\nStructural feature columns: {list(struct_features.columns[1:])}")

# Preview files with repeated UpdateResidentValue
repeated_urv = struct_features[struct_features['repeated_update_resident_value']]
print(f"Files with repeated UpdateResidentValue: {len(repeated_urv):,}")


Aggregating structural NTFS features per file...

Structural feature columns: ['only_SI_modified', 'num_update_resident_value', 'repeated_update_resident_value', 'consecutive_timestamp_changes', 'num_logfile_events']
Files with repeated UpdateResidentValue: 5,206


In [27]:
# [Cell 10] Aggregate Cross-Artifact Features (Category C)
# Reference: Oh et al. Algorithms 5-7 (UsnJrnl-1A)

def aggregate_cross_artifact_features(group):
    """
    Aggregate cross-artifact consistency features.
    
    Algorithm 5 references:
    - BASIC_INFO_CHANGE + CLOSE is a corroborative detection pattern
    - Absence of USN evidence does NOT invalidate LogFile-based detection

    Algorithm 6 references:
    - Compare $SI-C with FILE_CREATE event time
    - Compare last_basic_detection_pattern_time with $SI-E
    """
    logfile_events = group[group['is_logfile_event']]
    usnjrnl_events = group[group['is_usnjrnl_event']]
    
    ts_change_events = group[group['IsTimestampChange'] == True]
    basic_info_events = group[group['HasBasicInfoChange'] == True]
    close_events = group[group['HasClose'] == True]
    create_events = group[group['HasFileCreate'] == True]
    
    features = {
        # C.1 Has LogFile timestamp change
        'has_logfile_ts_change': len(ts_change_events) > 0,
        
        # C.2 Has USN BASIC_INFO_CHANGE (Algorithm 5)
        'has_usn_basic_info': len(basic_info_events) > 0,
        
        # C.3 Has USN CLOSE event
        'has_usn_close': len(close_events) > 0,
        
        # C.4 Has USN FILE_CREATE event
        'has_usn_file_create': len(create_events) > 0,
        
        # C.5 Count of each USN pattern
        'num_usn_basic_info': len(basic_info_events),
        'num_usn_close': len(close_events),
        'num_usn_file_create': len(create_events),
        
        # C.6 LogFile-USN mismatch detection
        # Timestamp change in LogFile but no BASIC_INFO_CHANGE in UsnJrnl
        'logfile_usn_mismatch': (
            len(ts_change_events) > 0 and len(basic_info_events) == 0
        ),
        
        # C.7 USN basic detection pattern (Algorithm 5)
        # BASIC_INFO_CHANGE followed by CLOSE
        'has_usn_basic_pattern': (
            len(basic_info_events) > 0 and len(close_events) > 0
        ),
        
        # C.8 Total USN events
        'num_usnjrnl_events': len(usnjrnl_events),
    }
    
    return pd.Series(features)

print("Aggregating cross-artifact features per file...")
cross_features = df.groupby('FileFRN').apply(aggregate_cross_artifact_features).reset_index()

print(f"\nCross-artifact feature columns: {list(cross_features.columns[1:])}")

# Preview files with mismatch
mismatch_files = cross_features[cross_features['logfile_usn_mismatch']]
print(f"Files with LogFile-USN mismatch: {len(mismatch_files):,}")


Aggregating cross-artifact features per file...

Cross-artifact feature columns: ['has_logfile_ts_change', 'has_usn_basic_info', 'has_usn_close', 'has_usn_file_create', 'num_usn_basic_info', 'num_usn_close', 'num_usn_file_create', 'logfile_usn_mismatch', 'has_usn_basic_pattern', 'num_usnjrnl_events']
Files with LogFile-USN mismatch: 11,872


In [28]:
# [Cell 11] Aggregate Temporal Behavior Features (Category D)

def aggregate_temporal_features(group):
    """
    Aggregate temporal behavior features.
    
    These features capture timing patterns that may indicate:
    - Automated timestomping tools (very fast operations)
    - Manual manipulation (longer gaps)
    - Burst activity (multiple changes in short window)
    """
    # Sort by EventTimestamp
    sorted_group = group.sort_values('EventTimestamp')
    timestamps = sorted_group['EventTimestamp'].dropna()
    
    features = {
        'min_inter_event_delta': np.nan,
        'max_inter_event_delta': np.nan,
        'mean_inter_event_delta': np.nan,
        'burstiness_score': 0.0,
        'event_time_span_seconds': np.nan,
    }
    
    if len(timestamps) >= 2:
        # Compute inter-event deltas
        deltas = timestamps.diff().dropna()
        delta_seconds = deltas.dt.total_seconds()
        
        features['min_inter_event_delta'] = delta_seconds.min()
        features['max_inter_event_delta'] = delta_seconds.max()
        features['mean_inter_event_delta'] = delta_seconds.mean()
        
        # Total time span
        features['event_time_span_seconds'] = (
            timestamps.max() - timestamps.min()
        ).total_seconds()
        
        # Burstiness: proportion of events within 1 second of each other
        burst_count = (delta_seconds <= 1.0).sum()
        features['burstiness_score'] = burst_count / len(delta_seconds) if len(delta_seconds) > 0 else 0
    
    return pd.Series(features)

print("Aggregating temporal behavior features per file...")
temporal_features = df.groupby('FileFRN').apply(aggregate_temporal_features).reset_index()

print(f"\nTemporal feature columns: {list(temporal_features.columns[1:])}")

# Preview high burstiness files
bursty_files = temporal_features[temporal_features['burstiness_score'] > 0.5]
print(f"Files with high burstiness (>0.5): {len(bursty_files):,}")


Aggregating temporal behavior features per file...

Temporal feature columns: ['min_inter_event_delta', 'max_inter_event_delta', 'mean_inter_event_delta', 'burstiness_score', 'event_time_span_seconds']
Files with high burstiness (>0.5): 17,728


In [29]:
# [Cell 12] Merge All Feature Categories

# Get file metadata
file_metadata = df.groupby('FileFRN').agg({
    'dataID': 'first',
    'FileName': 'first',
    'FilePath': 'first',
}).reset_index()

# Merge all feature sets
print("Merging all feature categories...")

file_features = file_metadata.merge(ts_features, on='FileFRN', how='left')
file_features = file_features.merge(struct_features, on='FileFRN', how='left')
file_features = file_features.merge(cross_features, on='FileFRN', how='left')
file_features = file_features.merge(temporal_features, on='FileFRN', how='left')

# Fill NaN values appropriately
numeric_cols = file_features.select_dtypes(include=[np.number]).columns
file_features[numeric_cols] = file_features[numeric_cols].fillna(0)

bool_cols = file_features.select_dtypes(include=[bool]).columns
file_features[bool_cols] = file_features[bool_cols].fillna(False)

print(f"\nFinal feature DataFrame shape: {file_features.shape}")
print(f"Total files: {len(file_features):,}")
print(f"Total features: {len(file_features.columns) - 4}")  # Exclude metadata columns

# Show feature statistics
print("\n--- Feature Statistics ---")
feature_cols = [c for c in file_features.columns if c not in ['FileFRN', 'dataID', 'FileName', 'FilePath']]
for col in feature_cols[:10]:  # Show first 10
    if file_features[col].dtype == bool:
        print(f"{col}: {file_features[col].sum():,} True")
    else:
        print(f"{col}: min={file_features[col].min():.2f}, max={file_features[col].max():.2f}, mean={file_features[col].mean():.2f}")


Merging all feature categories...

Final feature DataFrame shape: (109985, 32)
Total files: 109,985
Total features: 28

--- Feature Statistics ---
num_timestamp_changes: min=0.00, max=1002.00, mean=0.24
num_backward_jumps: min=0.00, max=27.00, mean=0.00
num_forward_jumps: min=0.00, max=998.00, mean=0.07
num_creation_changes: min=0.00, max=27.00, mean=0.00
max_backward_jump_seconds: min=0.00, max=416743032.99, mean=115240.93
mean_jump_seconds: min=0.00, max=416743032.97, mean=87835.47
timestamp_change_density: min=0.00, max=1.00, mean=0.09
num_zero_nanosecond_events: min=0.00, max=1.00, mean=0.00
only_SI_modified: 10,511 True
num_update_resident_value: min=0.00, max=2025.00, mean=0.36


## Detection Flags (Logic-Based, NOT ML)

Each flag is derived from computed features and references a specific Oh et al. detection rule.

### Flag Definitions

| Flag | Logic | Oh et al. Reference |
|------|-------|---------------------|
| `flag_backward_timestamp` | num_backward_jumps > 0 | Algorithm 3: before_$SI-M > after_$SI-M |
| `flag_creation_changed` | num_creation_changes > 0 | Algorithm 3: before_$SI-C != after_$SI-C |
| `flag_zero_nanoseconds` | num_zero_nanosecond_events > 0 | Algorithm 3: after_$SI-*.nanosecond == 0 |
| `flag_logfile_usn_mismatch` | logfile_usn_mismatch == True | Algorithms 5-7: Cross-artifact consistency |
| `flag_usn_basic_pattern` | has_usn_basic_pattern == True | Algorithm 5: BASIC_INFO_CHANGE + CLOSE |
| `flag_repeated_si_update` | repeated_update_resident_value == True | Algorithm 1: Multiple $SI updates |
| `flag_potential_timestomp` | (backward OR zero_ns) AND ts_changes > 0 | Combined: Core detection indicators |
| `flag_silent_timestomp` | logfile_ts_change AND NOT usn_basic_info | Cross-artifact: Change without USN evidence |


In [30]:
# [Cell 14] Compute Detection Flags
# Reference: Oh et al. Core Logic + Additional Validation

print("Computing detection flags...")

detection_flags = file_features[['FileFRN', 'dataID', 'FileName', 'FilePath']].copy()

# Flag 1: Backward timestamp jump detected (Algorithm 3)
# "if event.before_$SI-M > event.after_$SI-M then detection_target <- TRUE"
detection_flags['flag_backward_timestamp'] = file_features['num_backward_jumps'] > 0

# Flag 2: Creation time changed (Algorithm 3)
# "if event.before_$SI-C != event.after_$SI-C then detection_target <- TRUE"
detection_flags['flag_creation_changed'] = file_features['num_creation_changes'] > 0

# Flag 3: Zero nanoseconds detected (Algorithm 3 additional factor)
# "if event.after_$SI-C.nanosecond == 0 then additional_detection_factor <- TRUE"
detection_flags['flag_zero_nanoseconds'] = file_features['num_zero_nanosecond_events'] > 0

# Flag 4: LogFile-UsnJrnl mismatch (Algorithms 5-7 cross-artifact)
# Timestamp change in LogFile but no BASIC_INFO_CHANGE in UsnJrnl
detection_flags['flag_logfile_usn_mismatch'] = file_features['logfile_usn_mismatch']

# Flag 5: USN basic detection pattern (Algorithm 5)
# BASIC_INFO_CHANGE followed by CLOSE within time window
detection_flags['flag_usn_basic_pattern'] = file_features['has_usn_basic_pattern']

# Flag 6: Repeated $SI updates (Algorithm 1)
# Multiple UpdateResidentValue operations on same file
detection_flags['flag_repeated_si_update'] = file_features['repeated_update_resident_value']

# Flag 7: Only $SI modified (Algorithm 10/11 - $FN checker prerequisite)
# No $FN timestamps changed, only $SI
detection_flags['flag_only_si_modified'] = file_features['only_SI_modified']

# Flag 8: Potential timestomp (Combined core logic)
# File has timestamp changes AND (backward jump OR zero nanoseconds)
detection_flags['flag_potential_timestomp'] = (
    (file_features['num_timestamp_changes'] > 0) &
    (
        (file_features['num_backward_jumps'] > 0) |
        (file_features['num_zero_nanosecond_events'] > 0)
    )
)


# Flag 9: Silent timestomp (Cross-artifact anomaly)
# LogFile shows timestamp change but no USN BASIC_INFO_CHANGE evidence
detection_flags['flag_silent_timestomp'] = (
    (file_features['has_logfile_ts_change']) &
    (~file_features['has_usn_basic_info'])
)


# Flag 10: High suspicion (multiple indicators)
detection_flags['flag_high_suspicion'] = (
    (file_features['num_timestamp_changes'] >= 1) &
    (
        (file_features['num_backward_jumps'] > 0) |
        (file_features['num_zero_nanosecond_events'] > 0) |
        (file_features['repeated_update_resident_value'])
    )
)


print("\n--- Detection Flag Summary ---")
flag_cols = [c for c in detection_flags.columns if c.startswith('flag_')]
for col in flag_cols:
    count = detection_flags[col].sum()
    print(f"{col}: {count:,} files flagged")


Computing detection flags...

--- Detection Flag Summary ---
flag_backward_timestamp: 58 files flagged
flag_creation_changed: 36 files flagged
flag_zero_nanoseconds: 31 files flagged
flag_logfile_usn_mismatch: 11,872 files flagged
flag_usn_basic_pattern: 43 files flagged
flag_repeated_si_update: 5,206 files flagged
flag_only_si_modified: 10,511 files flagged
flag_potential_timestomp: 58 files flagged
flag_silent_timestomp: 11,872 files flagged
flag_high_suspicion: 5,155 files flagged


In [31]:
# [Cell 15] Identify Most Suspicious Files

# Compute suspicion score (count of true flags)
flag_cols = [c for c in detection_flags.columns if c.startswith('flag_')]
detection_flags['suspicion_score'] = detection_flags[flag_cols].sum(axis=1)
detection_flags['weighted_suspicion_score'] = (
    3 * detection_flags['flag_potential_timestomp'].astype(int) +
    2 * detection_flags['flag_backward_timestamp'].astype(int) +
    2 * detection_flags['flag_zero_nanoseconds'].astype(int) +
    2 * detection_flags['flag_repeated_si_update'].astype(int) +
    1 * detection_flags['flag_silent_timestomp'].astype(int) +
    1 * detection_flags['flag_usn_basic_pattern'].astype(int)
)


# Merge with key features for analysis
suspicious_analysis = detection_flags.merge(
    file_features[['FileFRN', 'num_timestamp_changes', 'num_backward_jumps', 
                   'max_backward_jump_seconds', 'num_zero_nanosecond_events']],
    on='FileFRN'
)

# Sort by suspicion score
suspicious_analysis = suspicious_analysis.sort_values('suspicion_score', ascending=False)

# Show top 20 most suspicious files
print("=== TOP 20 MOST SUSPICIOUS FILES ===\n")
top_suspicious = suspicious_analysis.head(20)

display_cols = ['FileFRN', 'FileName', 'suspicion_score', 
                'num_timestamp_changes', 'num_backward_jumps', 
                'max_backward_jump_seconds', 'num_zero_nanosecond_events',
                'flag_potential_timestomp', 'flag_high_suspicion']

print(top_suspicious[display_cols].to_string())

# Statistics
print(f"\n--- Suspicion Score Distribution ---")
print(suspicious_analysis['suspicion_score'].describe())

# Files with highest suspicion
high_suspicion = suspicious_analysis[suspicious_analysis['suspicion_score'] >= 3]
print(f"\nFiles with suspicion score >= 3: {len(high_suspicion):,}")


=== TOP 20 MOST SUSPICIOUS FILES ===

       FileFRN                             FileName  suspicion_score  num_timestamp_changes  num_backward_jumps  max_backward_jump_seconds  num_zero_nanosecond_events  flag_potential_timestomp  flag_high_suspicion
11035  49180.0  NewFileTime_SI_MAC_Manipulation.dll                8                    5.0                 1.0               3.153599e+07                         1.0                      True                 True
10999  49123.0                          polish2.dat                7                    2.0                 1.0               4.167430e+08                         1.0                      True                 True
11013  49155.0                             user.txt                7                    2.0                 1.0               4.167321e+08                         1.0                      True                 True
11006  49139.0                         timezone.dat                7                    2.0               

In [32]:
# [Cell 16] Validate Against Known Patterns
# Ground truth for 12-Kimsuky: boof.dll, boof.exe, boof.sys

print("=== VALIDATION: Searching for Known Malicious Patterns ===\n")

# The ground truth mentions timestamps changed from 2023-01-05 to 2019-12-07
# with zero nanoseconds

# Find files with large backward jumps (more than 1 year = ~31M seconds)
large_backward = file_features[file_features['max_backward_jump_seconds'] > 31536000]
print(f"Files with backward jumps > 1 year: {len(large_backward):,}")

if len(large_backward) > 0:
    print("\nFiles with large backward jumps:")
    print(large_backward[['FileFRN', 'FileName', 'FilePath', 
                          'max_backward_jump_seconds', 'num_backward_jumps']].to_string())

# Find files with both backward jumps AND zero nanoseconds
combined_indicators = file_features[
    (file_features['num_backward_jumps'] > 0) & 
    (file_features['num_zero_nanosecond_events'] > 0)
]
print(f"\nFiles with backward jumps AND zero nanoseconds: {len(combined_indicators):,}")

if len(combined_indicators) > 0:
    print("\nTop 10 files with combined indicators:")
    print(combined_indicators[['FileFRN', 'FileName', 'FilePath',
                               'num_backward_jumps', 'num_zero_nanosecond_events',
                               'max_backward_jump_seconds']].head(10).to_string())


=== VALIDATION: Searching for Known Malicious Patterns ===

Files with backward jumps > 1 year: 31

Files with large backward jumps:
       FileFRN                                 FileName                                                                             FilePath  max_backward_jump_seconds  num_backward_jumps
970     7230.0                         Windows Mail.lnk           /Users/blueangel/AppData/Roaming/Microsoft/Windows/Recent/Windows Mail.lnk               3.197590e+07                 4.0
10758  48733.0                    Boot Sector FAT32.tpl                    /Users/blueangel/AppData/Local/Temp/RarSFX1/Boot Sector FAT32.tpl               4.167430e+08                 1.0
10803  48806.0                      Boot Sector FAT.tpl                      /Users/blueangel/AppData/Local/Temp/RarSFX1/Boot Sector FAT.tpl               4.167430e+08                 1.0
10939  49023.0                     Boot Sector NTFS.tpl                     /Users/blueangel/AppData/Local/Temp/Rar

In [33]:
# [Cell 17] Detailed Analysis of Top Suspicious Files

# Get top 5 most suspicious files
top_5_frns = suspicious_analysis.head(5)['FileFRN'].tolist()

print("=== DETAILED EVENT ANALYSIS FOR TOP 5 SUSPICIOUS FILES ===\n")

for frn in top_5_frns:
    file_events = df[df['FileFRN'] == frn]
    ts_events = file_events[file_events['IsTimestampChange'] == True]
    
    file_info = file_features[file_features['FileFRN'] == frn].iloc[0]
    
    print(f"{'='*60}")
    print(f"FileFRN: {frn}")
    print(f"FileName: {file_info['FileName']}")
    print(f"FilePath: {file_info['FilePath']}")
    print(f"{'='*60}")
    
    print(f"\nTotal events: {len(file_events)}")
    print(f"Timestamp change events: {len(ts_events)}")
    print(f"Backward jumps: {file_info['num_backward_jumps']}")
    print(f"Zero nanosecond events: {file_info['num_zero_nanosecond_events']}")
    
    if len(ts_events) > 0:
        print("\n--- Timestamp Change Events ---")
        ts_display = ts_events[['LSN', 'Undo_$SI-M', 'Redo_$SI-M', 
                                'Undo_$SI-C', 'Redo_$SI-C']].head(5)
        print(ts_display.to_string())
    
    print("\n")


=== DETAILED EVENT ANALYSIS FOR TOP 5 SUSPICIOUS FILES ===

FileFRN: 49180.0
FileName: NewFileTime_SI_MAC_Manipulation.dll
FilePath: /Program Files/Windows Mail/NewFileTime_SI_MAC_Manipulation.dll

Total events: 161
Timestamp change events: 5
Backward jumps: 1.0
Zero nanosecond events: 1.0

--- Timestamp Change Events ---
                 LSN                 Undo_$SI-M          Redo_$SI-M                 Undo_$SI-C          Redo_$SI-C
169639  1.005299e+10                        NaT                 NaT                        NaT                 NaT
169646  1.005446e+10                        NaT                 NaT                        NaT                 NaT
169672  1.005480e+10                        NaT                 NaT                        NaT                 NaT
169673  1.005493e+10                        NaT                 NaT                        NaT                 NaT
169674  1.005493e+10 2023-12-25 16:36:29.782546 2022-12-25 16:36:44 2023-12-25 16:36:29.406054 2022-1

In [34]:
# [Cell 18] Export File Features CSV

# Select columns for export (exclude raw identifiers except FileFRN)
export_features = file_features.copy()

# Ensure proper column ordering
metadata_cols = ['dataID', 'FileFRN', 'FileName', 'FilePath']
feature_cols = [c for c in export_features.columns if c not in metadata_cols]
export_features = export_features[metadata_cols + feature_cols]

# Export
export_features.to_csv(FEATURES_OUTPUT, index=False, encoding='utf-8')

print(f"=== Features CSV Exported ===")
print(f"Output: {FEATURES_OUTPUT}")
print(f"Rows: {len(export_features):,}")
print(f"Columns: {len(export_features.columns)}")
print(f"File size: {FEATURES_OUTPUT.stat().st_size / 1024:.2f} KB")

print("\nColumn list:")
for i, col in enumerate(export_features.columns, 1):
    print(f"  {i:2d}. {col}")


=== Features CSV Exported ===
Output: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 3: Feature Engineering & Labeling/Test Notebook Outputs/phase3_file_features_03-PE.csv
Rows: 109,985
Columns: 32
File size: 35761.05 KB

Column list:
   1. dataID
   2. FileFRN
   3. FileName
   4. FilePath
   5. num_timestamp_changes
   6. num_backward_jumps
   7. num_forward_jumps
   8. num_creation_changes
   9. max_backward_jump_seconds
  10. mean_jump_seconds
  11. timestamp_change_density
  12. num_zero_nanosecond_events
  13. only_SI_modified
  14. num_update_resident_value
  15. repeated_update_resident_value
  16. consecutive_timestamp_changes
  17. num_logfile_events
  18. has_logfile_ts_change
  19. has_usn_basic_info
  20. has_usn_close
  21. has_usn_file_create
  22. num_usn_basic_info
  23. num_usn_close
  24. num_usn_file_create
  25. logfile_usn_mismatch
  26. has_usn_basic_pattern
  27. num_usnjrnl_events
  28. min_inter_event_delta
  29. max_inter_event_delta
  30. mean_inter

In [35]:
# [Cell 19] Export Detection Flags CSV

# Select columns for export
export_flags = detection_flags.copy()

# Add suspicion score to flags output
export_flags['suspicion_score'] = suspicious_analysis['suspicion_score'].values

# Ensure proper column ordering
metadata_cols = ['dataID', 'FileFRN', 'FileName', 'FilePath']
flag_cols = [c for c in export_flags.columns if c.startswith('flag_')]
other_cols = ['suspicion_score']
export_flags = export_flags[metadata_cols + flag_cols + other_cols]

# Export
export_flags.to_csv(FLAGS_OUTPUT, index=False, encoding='utf-8')

print(f"=== Detection Flags CSV Exported ===")
print(f"Output: {FLAGS_OUTPUT}")
print(f"Rows: {len(export_flags):,}")
print(f"Columns: {len(export_flags.columns)}")
print(f"File size: {FLAGS_OUTPUT.stat().st_size / 1024:.2f} KB")

print("\nColumn list:")
for i, col in enumerate(export_flags.columns, 1):
    print(f"  {i:2d}. {col}")


=== Detection Flags CSV Exported ===
Output: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 3: Feature Engineering & Labeling/Test Notebook Outputs/phase3_detection_flags_03-PE.csv
Rows: 109,985
Columns: 15
File size: 29520.34 KB

Column list:
   1. dataID
   2. FileFRN
   3. FileName
   4. FilePath
   5. flag_backward_timestamp
   6. flag_creation_changed
   7. flag_zero_nanoseconds
   8. flag_logfile_usn_mismatch
   9. flag_usn_basic_pattern
  10. flag_repeated_si_update
  11. flag_only_si_modified
  12. flag_potential_timestomp
  13. flag_silent_timestomp
  14. flag_high_suspicion
  15. suspicion_score


## Summary

### Phase 3 Processing Complete for 12-Kimsuky Dataset

**Input:** grouped_events_12-Kimsuky.csv (from Phase 2)

**Outputs:**
1. `phase3_file_features_12_Kimsuky.csv` - File-level features
2. `phase3_detection_flags_12_Kimsuky.csv` - Logic-based detection flags

### Features Computed (per Oh et al. algorithms)
- **Timestamp Change Features**: num_timestamp_changes, num_backward_jumps, etc.
- **Structural NTFS Patterns**: only_SI_modified, repeated_update_resident_value
- **Cross-Artifact Consistency**: has_logfile_ts_change, logfile_usn_mismatch
- **Temporal Behavior**: min_inter_event_delta, burstiness_score

### Detection Flags Applied
- flag_backward_timestamp (Algorithm 3)
- flag_creation_changed (Algorithm 3)
- flag_zero_nanoseconds (Algorithm 3)
- flag_logfile_usn_mismatch (Algorithms 5-7)
- flag_usn_basic_pattern (Algorithm 5)
- flag_potential_timestomp (Combined)
- flag_high_suspicion (Combined)

### Next Steps
1. Phase 3 Part 2: Compare with LogTracker ground truth (Suspicious.txt)
2. Apply same logic to all training/validation datasets
3. Phase 4: Use features for ML model training


In [36]:
# [Cell 21] Final Statistics

print("=" * 60)
print("PHASE 3 COMPLETE: 12-Kimsuky Dataset")
print("=" * 60)

print(f"\n{'Metric':<45} {'Value':>12}")
print("-" * 60)
print(f"{'Total files analyzed':<45} {len(file_features):>12,}")
print(f"{'Files with timestamp changes':<45} {(file_features['num_timestamp_changes'] > 0).sum():>12,}")
print(f"{'Files with backward jumps':<45} {(file_features['num_backward_jumps'] > 0).sum():>12,}")
print(f"{'Files with zero nanoseconds':<45} {(file_features['num_zero_nanosecond_events'] > 0).sum():>12,}")
print(f"{'Files flagged: potential_timestomp':<45} {detection_flags['flag_potential_timestomp'].sum():>12,}")
print(f"{'Files flagged: high_suspicion':<45} {detection_flags['flag_high_suspicion'].sum():>12,}")
print(f"{'Files with suspicion_score >= 3':<45} {(suspicious_analysis['suspicion_score'] >= 3).sum():>12,}")
print("-" * 60)

print("\nPhase 3 Part 1 complete. Ready for ground truth validation in Part 2.")


PHASE 3 COMPLETE: 12-Kimsuky Dataset

Metric                                               Value
------------------------------------------------------------
Total files analyzed                               109,985
Files with timestamp changes                        11,895
Files with backward jumps                               58
Files with zero nanoseconds                             31
Files flagged: potential_timestomp                      58
Files flagged: high_suspicion                        5,155
Files with suspicion_score >= 3                     11,808
------------------------------------------------------------

Phase 3 Part 1 complete. Ready for ground truth validation in Part 2.
